# Knowledge Graphs v1

### 2. Neo4J & Jupyter Environment
This notebook needs an instance of [Neo4j](https://www.neo4j.com) to talk to. I used docker to run Neo4J locally using the following command:
```
docker run --name testneo4j -p7474:7474 -p7687:7687 -d \
    -v $HOME/neo4j/data:/data \
    -v $HOME/neo4j/logs:/logs \
    -v $HOME/neo4j/import:/var/lib/neo4j/import \
    -v $HOME/neo4j/plugins:/plugins \
    --env NEO4J_AUTH=neo4j/password \
    neo4j:latest
```
**Note:** No particular plugins are needed. 

You can also use a Neo4J Aurora instance. 

#### Jupyter Environment
Regardless of how you run Neo4J. You need to set some environment variables in the notebook's environment:

| Variable | Description | Value for above Docker |
|----------|-------------|------------------------|
| NEO4J_URL | Where to find the instance of Neo4j. | bolt://localhost:7687 |
| NEO4J_USER | The username for the database. | neo4j |
| NEO4J_PASSWORD | The password for the database. | password |


### 3. Synthetic data and working directory
The data I used for this notebook came from [Synthea](https://synthea.mitre.org/). Using the 

All the questions here us the FHIR Bundle: `fhir_data/stanfor_llm_on_fhir`

In [1]:
# Imports needed

import glob
import json
import os
import re

from helpers.FHIR_to_graph_v1 import resource_to_edges, resource_to_node

# Imports from other local python files
from helpers.neo4j_graph import Graph

## Establish Database Connection

The cell connects to the Neo4J instance. It relies on several environment variables. 

**PLEASE NOTE**: The variable have been changed to support multiple databases in the same instance. 

| Variable            | Description                          | Sample Value          |
|---------------------|--------------------------------------|-----------------------|
| FHIR_GRAPH_URL      | Where to find the instance of Neo4j. | bolt://localhost:7687 |
| FHIR_GRAPH_USER     | The username for the database.       | neo4j                 |
| FHIR_GRAPH_PASSWORD | The password for the database.       | password              |
| FHIR_GRAPH_DATABASE | The name of the database instance.   | neo4j                 |

In [9]:
NEO4J_URI = os.getenv("FHIR_GRAPH_URL", "neo4j://localhost:7687")
USERNAME = os.getenv("FHIR_GRAPH_USER", "neo4j")
PASSWORD = os.getenv("FHIR_GRAPH_PASSWORD", "password")
DATABASE = os.getenv("FHIR_GRAPH_DATABASE", "neo4j")

graph = Graph(NEO4J_URI, USERNAME, PASSWORD, DATABASE)

## Helper Database Cells

The following three cells are here to be used to manage the database. They do not need to be run on a blank database. 

http://localhost:7474/browser/

In [10]:
print(graph.resource_metrics())

[]


In [11]:
print(graph.database_metrics())

(0, 0)


In [12]:
graph.wipe_database()

'Deleted 0 nodes and 0 relationships in 0.011 seconds'

## Load FHIR into the Graph

This cell opens the bundle and creates the nodes and edges in the graph for each resource. 

Every resource will result in a node that has a label based on the resource type and as a `resource`. The values within the resource will be flattened 
into properties within the node. Also, a property called `text` will include a string representation of the resource. 

Additionally, nodes will be created for every unique date (ignoring time) found in the FHIR resources. 

Edges will be created for every reference in the resource to something that can be found within the bundles loaded. So the linking resource doesn't have 
to be in the same bundle, but it must be in a bundle that is loaded. 

Edges will also connect resources to the dates found inside them. 

**Warning:** This cell may take sometime to run. 

In [12]:
# synthea_bundles = glob.glob("/home/baptvit/Documents/github/mestrado/fhir-rag/fhir_rag/fhir_data/sythea_fhir/Alfonso758_Bins636_e80d4c62-149a-a6a6-4b39-9d4aa3e07ba7.json")
synthea_bundles = glob.glob(
    "/home/baptvit/repositories/fhir_based_gen_ai_research/fhir_rag/fhir_data/stanford_llm_on_fhir/Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58.json"
)
synthea_bundles = synthea_bundles[0:1]
synthea_bundles.sort()
synthea_bundles

['/home/baptvit/repositories/fhir_based_gen_ai_research/fhir_rag/fhir_data/stanford_llm_on_fhir/Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58.json']

In [ ]:
import time

In [14]:
start = time.time()

nodes = []
edges = []
dates = set()  # set is used here to make sure dates are unique
for bundle_file_name in synthea_bundles:
    with open(bundle_file_name) as raw:
        bundle = json.load(raw)
        for entry in bundle["entry"]:
            resource_type = entry["resource"]["resourceType"]
            if resource_type != "Provenance":
                # generated the cypher for creating the resource node
                nodes.append(
                    resource_to_node(
                        entry["resource"],
                        bundle_file_name.split("/")[-1].replace(".json", ""),
                    )
                )
                # generated the cypher for creating the reference & date edges and capture dates
                node_edges, node_dates = resource_to_edges(entry["resource"])
                edges += node_edges
                dates.update(node_dates)

print("hello")
end = time.time()
print(end - start)

beatris270 bogan287
encounter
condition
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
procedure
immunization
diagnosticreport
diagnosticreport
documentreference
claim
explanationofbenefit
encounter
condition
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
procedure
immunization
diagnosticreport
documentreference
claim
explanationofbenefit
encounter
condition
observation
observation
observation
observation
observation
observation
observation
observation
observation
observation
immunization
immunization
immunization
immunization
immunization
diagnosticreport
documentreference
claim
explanationofbenefit
encounter
condition
careteam
careplan
diagnosticreport
documentreference
claim
explanationofbenefit
encounter
obse

Create the nodes for each resource

In [15]:
# create the nodes for resources
for node in nodes:
    try:
        graph.query(node)
    except:
        print(node + "\n\n")

CREATE (:DiagnosticReport:resource {name: "DiagnosticReport", consumer_id: "Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58", text: 'The Diagnostic Report with ID d37cd98d-fe92-a3c9-dee3-0bc00000f431 is classified as a final report. It includes categories for "History and physical note" and "Evaluation + Plan note." The subject of the report is referenced by the UUID 5b3645de-a2d0-d016-0839-bab3757c4c58, and it is linked to an encounter with the UUID d7a2721e-1568-274b-64bb-58c2ec026b65. The report was effective on June 7, 2015, at 13:37:42 UTC and was issued at 13:37:42.868 UTC the same day. Dr. Elisa944 Rojo930, identified by NPI 9999950899, is the performer of this report. The report includes a presented form in plain text format containing medical history and patient details, such as the patient's condition and treatment plan, as well as additional observations and recommendations.', resource_type: "DiagnosticReport",id: "d37cd98d-fe92-a3c9-dee3-0bc00000f431",meta_profile_

Create nodes for the import dates

In [16]:
date_pattern = re.compile(r"([0-9]+)/([0-9]+)/([0-9]+)")

# create the nodes for dates
for date in dates:
    date_parts = date_pattern.findall(date)[0]
    cypher_date = f"{date_parts[2]}-{date_parts[0]}-{date_parts[1]}"
    cypher = (
        'CREATE (:Date {name:"'
        + date
        + '", id: "'
        + date
        + '", date: date("'
        + cypher_date
        + '"), text:"'
        + date
        + '"})'
    )
    graph.query(cypher)

Create the edges/relationships between each resources

In [17]:
# create the edges
for edge in edges:
    try:
        graph.query(edge)
    except:
        print(f"Failed to create edge: {edge}")

In [ ]:
edges[1]

## Create the Vector Embedding Index in the Graph

This cell creates a Vector Index in Neo4J. It looks at nodes labeled as `resource` and indexes the string representation in the `text` property. 

**Warning:** This cell may take sometime to run. 

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    print("Sorry no cuda.")

In [ ]:
device

In [ ]:
import os

os.environ["DIAL_KEY"] = ""
DIAL_KEY = ""


from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=os.getenv("DIAL_KEY"),
    api_version="2024-02-01",
    azure_endpoint="",
)


models = client.models.list()
models

client.embeddings

In [ ]:
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small-1",
    azure_deployment="text-embedding-3-small-1",
    api_key="",
    api_version="2023-08-01-preview",
    azure_endpoint="",
)

In [ ]:
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

Neo4jVector.from_existing_graph(
    embeddings,
    url=NEO4J_URI,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE,
    index_name="fhir_text",
    node_label="resource",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

In [18]:
from langchain.vectorstores.neo4j_vector import Neo4jVector

Neo4jVector.from_existing_graph(
    HuggingFaceBgeEmbeddings(model_name="BAAI/bge-small-en-v1.5"),
    url=NEO4J_URI,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE,
    index_name="fhir_text",
    node_label="resource",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

### Create Vector Index 

This cell creates a new vector index, using the index created above. 

This is here because running the cell above can take time and only should be done one time when the DB is created. 

In [19]:
vector_index = Neo4jVector.from_existing_index(
    HuggingFaceBgeEmbeddings(model_name="BAAI/bge-small-en-v1.5"),
    url=NEO4J_URI,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE,
    index_name="fhir_text",
)

# Testing the pre trasnformation of the FHIR 

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    api_key="",
    azure_deployment="gpt-4o-mini-2024-07-18",
    api_version="2023-08-01-preview",
    azure_endpoint="",
)

In [ ]:
response = llm.invoke("""
As a specialist in FHIR R4 conversion, transform the following FHIR R4 resource into a concise human-readable text non-formated, while preserving all essential details. Output should only include the final text: \"\"\"{"resourceType": "MedicationRequest", "id": "cf18362b-5057-8a39-039b-dbc9e13ed518", "meta": {"profile": ["http://hl7.org/fhir/us/core/StructureDefinition/us-core-medicationrequest"]}, "status": "active", "intent": "order", "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/medicationrequest-category", "code": "community", "display": "Community"}], "text": "Community"}], "medicationCodeableConcept": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "1870230", "display": "NDA020800 0.3 ML Epinephrine 1 MG/ML Auto-Injector"}], "text": "NDA020800 0.3 ML Epinephrine 1 MG/ML Auto-Injector"}, "subject": {"reference": "urn:uuid:5b3645de-a2d0-d016-0839-bab3757c4c58"}, "encounter": {"reference": "urn:uuid:0bc7f36f-9d73-bdef-bfaa-b097c4b99dbc"}, "authoredOn": "2017-08-30T12:14:48+00:00", "requester": {"reference": "Practitioner?identifier=http://hl7.org/fhir/sid/us-npi|9999990697", "display": "Dr. Alvin56 Crona259"}, "dosageInstruction": [{"sequence": 1, "text": "Take as needed.", "asNeededBoolean": true}]}\"\"\"
""")

In [ ]:
response.content

In [ ]:
from langchain_openai import AzureChatOpenAI


def preprocess_fhir_resouce(resource: str) -> str:
    """Pre process the input FHIR resource into a humam text format"""
    from langchain_openai import AzureChatOpenAI

    llm = AzureChatOpenAI(
        api_key="",
        azure_deployment="gpt-4o-mini-2024-07-18",
        api_version="2023-08-01-preview",
        azure_endpoint="",
    )
    response = llm.invoke(f"""
    As a specialist in FHIR R4 conversion, transform the following FHIR R4 resource into a concise human-readable text non-formated, while preserving all essential details. Output should only include the final text: \"\"\"{resource}\"\"\"
    """)
    return response.content